<a href="https://colab.research.google.com/github/beneducarrco99/Laboratorio1S1/blob/main/S2_Ejercicios_Resueltos_CDD_Carrasco_Benjamin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicios resueltos — Calidad y preparación del dato
**Ciencia de Datos · Semana 2 · Advance UCSC**

Trabajaremos sobre `S2_Dataset_Ejercicios_CDD.csv`: un registro de patentes comerciales municipales con **590 filas** y los ocho defectos que aparecen una y otra vez en datos reales.

**Cómo usar este cuaderno.** Cada ejercicio tiene tres partes: el enunciado, una celda vacía para que lo intentes, y la solución comentada. **Intenta primero.** El valor está en el intento, no en la respuesta.

Y una advertencia de método que vale para todo el curso: frente a cada defecto hay **tres decisiones legítimas** —corregir, descartar o declarar como limitación— y lo que se evalúa no es cuál elegiste, sino que la hayas justificado.

In [2]:
import pandas as pd
import numpy as np

RUTA = ''
SEMILLA = 20262
np.random.seed(SEMILLA)

df = pd.read_csv(RUTA + 'S2_Dataset_Ejercicios_CDD.csv', sep=';')
print('Forma:', df.shape)
df.head()

Forma: (590, 11)


,rol,establecimiento,rubro,comuna,region,fecha_patente,superficie_m2,patente_uf,trabajadores,estado,ultima_fiscalizacion
0,02-00286,La Bahía II,Panadería,Chillán,Ñuble,2016-06-19,74.8,"UF 10,58",2,Vigente,2018-11-29
1,04-00319,El Roble SpA,Panadería,Los Ángeles,Biobío,2022-10-06,79.1,10.94,2,Vigente,2024-03-30
2,01-00184,La Portada y Cía.,Farmacia,Chillán,Ñuble,2022-05-18,80.9,9.3,4,Vigente,2028-06-10
3,01-00029,Los Aromos Ltda.,Almacén,Concepción,Biobío,2018-08-05,13.2,2.24,1,Vigente,2025-06-23
4,04-00159,Los Maitenes SpA,Panadería,Concepción,Biobío,2025-07-01,14.3,2.11,1,Suspendida,2026-12-13


In [3]:
df.dtypes

,0
rol,object
establecimiento,object
rubro,object
comuna,object
region,object
fecha_patente,object
superficie_m2,float64
patente_uf,object
trabajadores,int64
estado,object


Antes de tocar nada: **cinco columnas que deberían ser numéricas o de fecha están como `object`**. Esa es la primera pista de que hay trabajo por delante.

---
## Ejercicio 1 · Completitud

**Enunciado.** Calcula cuántos valores faltan en cada columna y qué porcentaje representan. Luego responde: ¿los faltantes de `patente_uf` están repartidos al azar, o se concentran en algún grupo?

In [4]:
# Tu intento aquí
faltantes = df.isna().sum().to_frame('Faltantes')
faltantes['Porcentaje'] = (faltantes['Faltantes'] / len(df) * 100).round(2)
# faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending = False)

print(faltantes)

                      Faltantes  Porcentaje
rol                           0        0.00
establecimiento               0        0.00
rubro                         0        0.00
comuna                        0        0.00
region                        0        0.00
fecha_patente                 0        0.00
superficie_m2                 0        0.00
patente_uf                   21        3.56
trabajadores                  0        0.00
estado                        0        0.00
ultima_fiscalizacion         42        7.12


In [5]:
# Cruzamos con la columna "estado" para verificar en que grupo se concentran los faltantes de "patente_uf"
pd.crosstab(df['estado'], df['patente_uf'].isna(), normalize = 'index').round(2)

patente_uf,False,True
estado,,
Caducada,0.64,0.36
Suspendida,1.00,0.00
Vigente,1.00,0.00


### Solución

In [6]:
faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes[faltantes['faltantes'] > 0].sort_values('faltantes', ascending=False)

,faltantes,pct
ultima_fiscalizacion,42,7.12
patente_uf,21,3.56


In [7]:
# ¿Se concentran en algún grupo? Cruzamos con 'estado'
pd.crosstab(df['estado'], df['patente_uf'].isna(), normalize='index').round(3)

patente_uf,False,True
estado,,
Caducada,0.644,0.356
Suspendida,1.000,0.000
Vigente,1.000,0.000


**Lectura.** Los faltantes de `patente_uf` **no están al azar**: se concentran en las patentes `Caducada`. Eso cambia por completo qué corresponde hacer.

Si estuvieran repartidos al azar, imputar con la mediana sería defendible. Como se concentran en un grupo con significado —una patente caducada no tiene monto vigente—, **el faltante es informativo**: imputarlo inventaría montos que no existen y contaminaría cualquier comparación entre estados.

> **Decisión razonable:** conservar el faltante y excluir las caducadas de los cálculos de monto, declarándolo. *Bitácora: Declarar como limitación.*

---
## Ejercicio 2 · Valores centinela

**Enunciado.** Un valor centinela es un número real que en verdad significa «no hay dato». Este conjunto tiene tres, de tres sabores distintos. Encuéntralos en `trabajadores`, `superficie_m2` y `ultima_fiscalizacion`.

In [8]:
# Tu intento aquí
for col in df.select_dtypes(include = ['number', 'object']).columns:
  v = sorted(df[col].dropna().unique())
  print(col, v[:5], '...', v[-5:])

rol ['01 - 00171', '01 - 00508', '01-00001', '01-00005', '01-00007'] ... ['04-00542', '04-00547', '04-00550', '04-00551', '04-00553']
establecimiento ['  DON PEPE II ', '  EL MANZANO II ', '  EL MOLINO ', '  EL RINCÓN ', '  EL RINCÓN LTDA. '] ... ['Ñielol', 'Ñielol II', 'Ñielol Ltda.', 'Ñielol SpA', 'Ñielol y Cía.']
rubro ['ALMACÉN', 'Almacén', 'Bazar', 'Bazars', 'FARMACIA'] ... ['panadería', 'peluquería', 'restaurante', 'taller mecánico', 'verdulería']
comuna [' Chillán', ' Concepción', ' Coronel', 'CHILLÁN', 'Chillán'] ... ['chillán', 'concepción', 'coronel', 'san pedro de la paz', 'talcahuano']
region ['Biobío', 'Ñuble'] ... ['Biobío', 'Ñuble']
fecha_patente ['02/11/2021', '03/01/2025', '03/09/2018', '04-05-2024', '04/08/2021'] ... ['23-01-2018', '25/11/2021', '26/02/2016', '26/04/2023', '26/11/2019']
superficie_m2 [np.float64(0.0267), np.float64(0.0296), np.float64(0.0353), np.float64(0.0455), np.float64(0.0464)] ... [np.float64(3583.8), np.float64(3965.7), np.float64(4122.6), np.f

### Solución

In [9]:
trab = pd.to_numeric(df['trabajadores'], errors='coerce')
sup  = pd.to_numeric(df['superficie_m2'], errors='coerce')

print('trabajadores == -1      :', (trab == -1).sum())
print('superficie_m2 == 9999   :', (sup == 9999).sum())
print("ultima_fiscalizacion 'S/I':", (df['ultima_fiscalizacion'] == 'S/I').sum())

trabajadores == -1      : 19
superficie_m2 == 9999   : 10
ultima_fiscalizacion 'S/I': 17


In [10]:
# Por qué importa: el efecto sobre un promedio
print(f'Superficie promedio CON 9999 : {sup.mean():8.1f} m2')
print(f'Superficie promedio SIN 9999 : {sup[sup != 9999].mean():8.1f} m2')

Superficie promedio CON 9999 :    263.1 m2
Superficie promedio SIN 9999 :     95.3 m2


**Lectura.** Diez registros bastan para multiplicar el promedio. Y lo peligroso es que `superficie_m2` **es numérica**: el centinela no se delata en los tipos de dato. Solo aparece si miras los rangos.

> **Decisión razonable:** convertir los tres centinelas a `NaN` y tratarlos como faltantes. *Bitácora: Corregir (conversión a faltante).*

In [11]:
df['trabajadores']  = trab.replace(-1, np.nan)
df['superficie_m2'] = sup.replace(9999, np.nan)
df['ultima_fiscalizacion'] = df['ultima_fiscalizacion'].replace('S/I', np.nan)
print('Listo. Faltantes tras la conversión:')
print(df[['trabajadores', 'superficie_m2', 'ultima_fiscalizacion']].isna().sum())

Listo. Faltantes tras la conversión:
trabajadores            19
superficie_m2           10
ultima_fiscalizacion    59
dtype: int64


---
## Ejercicio 3 · Categorías inconsistentes

**Enunciado.** La columna `comuna` declara muchos más valores distintos de los que deberían existir. Averigua cuántos hay, por qué, y normalízala.

In [12]:
# Tu intento aquí
for col in ['comuna', 'region']:
  print(f'{col:7s} = {df[col].nunique():3d} valores distintos')

comuna  =  22 valores distintos
region  =   2 valores distintos


In [13]:
# Todos los valores en la columna "comuna"
df['comuna'].unique()

array(['Chillán', 'Los Ángeles', 'Concepción', ' Coronel', ' Concepción',
       'Talcahuano', 'San Pedro de la Paz', 'Coronel', 'TALCAHUANO',
       'Concepción ', 'san pedro de la paz', 'Concepcion', ' Chillán',
       'Los Ángeles ', 'CHILLÁN', 'Chillán ', 'Coronel ', 'concepción',
       'Talcahuano ', 'chillán', 'talcahuano', 'coronel'], dtype=object)

In [14]:
# Normalizamos los valores de "comuna"
EQUIVALENCIAS = [
    ' Chillán' : 'Chillán', 'CHILLÁN' : 'Chillán', 'Chillán ' : 'Chillán', 'chillán' : 'Chillán',
    'Los Ángeles ' : 'Los Ángeles',
    ' Concepción' : 'Concepción', 'Concepción ' : 'Concepción', 'Concepcion' : 'Concepción', 'concepción' : 'Concepción',
    ' Coronel' : 'Coronel', 'Coronel ' : 'Coronel', 'coronel' : 'Coronel',
    'san pedro de la paz' : 'San Pedro de la Paz',
    'TALCAHUANO' : 'Talcahuano', 'Talcahuno ' : 'Talcahuano', 'talcahuano' : 'Talcahuano'
    ]

df['comuna_norm'] = normalizar_texto(df['comuna']).replace(EQUIVALENCIAS)

SyntaxError: invalid syntax (267363197.py, line 3)

### Solución

In [15]:
print('Valores distintos declarados:', df['comuna'].nunique())
sorted(df['comuna'].unique())

Valores distintos declarados: 22


[' Chillán',
 ' Concepción',
 ' Coronel',
 'CHILLÁN',
 'Chillán',
 'Chillán ',
 'Concepcion',
 'Concepción',
 'Concepción ',
 'Coronel',
 'Coronel ',
 'Los Ángeles',
 'Los Ángeles ',
 'San Pedro de la Paz',
 'TALCAHUANO',
 'Talcahuano',
 'Talcahuano ',
 'chillán',
 'concepción',
 'coronel',
 'san pedro de la paz',
 'talcahuano']

Tres causas conviven: **mayúsculas/minúsculas**, **espacios sobrantes** y **tildes eliminadas**. La normalización tiene que atacar las tres.

In [16]:
def normalizar_texto(serie):
    """Quita espacios, unifica mayúsculas y elimina tildes."""
    return (serie.astype(str)
                 .str.strip()
                 .str.upper()
                 .str.normalize('NFKD')
                 .str.encode('ascii', 'ignore')
                 .str.decode('ascii'))

df['comuna_norm'] = normalizar_texto(df['comuna'])
df['rubro_norm']  = normalizar_texto(df['rubro'])

print('comuna :', df['comuna'].nunique(), '->', df['comuna_norm'].nunique())
print('rubro  :', df['rubro'].nunique(),  '->', df['rubro_norm'].nunique())

comuna : 22 -> 6
rubro  : 28 -> 14


In [17]:
# El rubro no queda perfecto: quedan plurales sueltos
sorted(df['rubro_norm'].unique())

['ALMACEN',
 'BAZAR',
 'BAZARS',
 'FARMACIA',
 'FERRETERIA',
 'LIBRERIA',
 'LIBRERIAS',
 'PANADERIA',
 'PANADERIAS',
 'PELUQUERIA',
 'RESTAURANTE',
 'TALLER MECANICO',
 'TALLER MECANICOS',
 'VERDULERIA']

**Lectura.** La normalización mecánica resuelve mayúsculas, espacios y tildes, pero **no resuelve los plurales** («ALMACEN» frente a «ALMACENS»). Eso exige una decisión humana: un diccionario de equivalencias explícito.

> Esta es la diferencia entre limpiar y *entender*. Ninguna función adivina que «ALMACENS» y «ALMACEN» son lo mismo; tú lo sabes porque conoces el dominio.

In [18]:
EQUIVALENCIAS = {'ALMACENS': 'ALMACEN', 'PANADERIAS': 'PANADERIA',
                 'RESTAURANTES': 'RESTAURANTE', 'FARMACIAS': 'FARMACIA',
                 'FERRETERIAS': 'FERRETERIA', 'LIBRERIAS': 'LIBRERIA',
                 'BAZARS': 'BAZAR', 'VERDULERIAS': 'VERDULERIA',
                 'PELUQUERIAS': 'PELUQUERIA', 'TALLER MECANICOS': 'TALLER MECANICO'}
df['rubro_norm'] = df['rubro_norm'].replace(EQUIVALENCIAS)
print('rubros finales:', df['rubro_norm'].nunique())
sorted(df['rubro_norm'].unique())

rubros finales: 10


['ALMACEN',
 'BAZAR',
 'FARMACIA',
 'FERRETERIA',
 'LIBRERIA',
 'PANADERIA',
 'PELUQUERIA',
 'RESTAURANTE',
 'TALLER MECANICO',
 'VERDULERIA']

---
## Ejercicio 4 · Duplicados exactos y casi-duplicados

**Enunciado.** Cuenta los duplicados exactos. Después busca los **casi-duplicados**: el mismo establecimiento cargado dos veces con distinto espaciado o capitalización.

In [21]:
# Tu intento aquí
print('Nro. de duplicados exactos: ', df.duplicated().sum())

Nro. de duplicados exactos:  17


In [24]:
# Para buscar casi-duplicados
def normalizar_texto(serie):
  return ( serie.astype(str)
                .str.strip() # Espacios sobrantes
                .str.upper() # Mayúsculas/minúsculas
                .str.normalize('NFKD')
                .str.encode('ascii', 'ignore')
                .str.decode('ascii') # Tildes
  )

clave = normalizar_texto(df['rol']) + ' | ' + normalizar_texto(df['establecimiento'])
df = df.loc[~clave.duplicated()].copy()

print(df)

          rol     establecimiento            rubro               comuna  \
0    02-00286         La Bahía II        Panadería              Chillán   
1    04-00319        El Roble SpA        Panadería          Los Ángeles   
2    01-00184   La Portada y Cía.         Farmacia              Chillán   
3    01-00029    Los Aromos Ltda.          Almacén           Concepción   
4    04-00159    Los Maitenes SpA        Panadería           Concepción   
..        ...                 ...              ...                  ...   
585  04-00441           Doña Rosa  Taller mecánico              Chillán   
586  01-00128          Ñielol SpA         Librería           Concepción   
587  02-00481     La Frontera SpA        Panadería  San Pedro de la Paz   
588  03-00384  Los Maitenes Ltda.         Librería          Los Ángeles   
589  03-00285     Las Camelias II          Almacén              Coronel   

     region fecha_patente  superficie_m2 patente_uf  trabajadores      estado  \
0     Ñuble    201

### Solución

In [25]:
print('Duplicados exactos (fila completa):', df.duplicated().sum())

Duplicados exactos (fila completa): 0


In [26]:
# Casi-duplicados: normalizamos la clave antes de comparar
clave = (df['rol'].astype(str).str.replace(' ', '', regex=False) + '|' +
         normalizar_texto(df['establecimiento']))
print('Registros repetidos tras normalizar la clave:', clave.duplicated().sum())

Registros repetidos tras normalizar la clave: 13


**Lectura.** Los duplicados exactos se ven con una línea. Los casi-duplicados **no**: hay que construir una clave normalizada primero. Son los que más daño hacen, porque sobreviven a cualquier `drop_duplicates()` ingenuo y inflan silenciosamente los conteos.

> **Decisión razonable:** eliminar duplicados sobre la clave normalizada, conservando el primer registro. *Bitácora: Corregir.*

In [27]:
antes = len(df)
df = df.loc[~clave.duplicated()].copy()
print(f'Filas: {antes} -> {len(df)}  ({antes - len(df)} eliminadas)')

Filas: 573 -> 560  (13 eliminadas)


---
## Ejercicio 5 · Numérico almacenado como texto

**Enunciado.** `patente_uf` quedó como `object`. Averigua qué contiene que no sea un número y conviértela.

In [30]:
# Tu intento aquí
# Ver que impide la conversión
no_numb = df.loc[pd.to_numeric(df['patente_uf'], errors = 'coerce').isna()
                 & df['patente_uf'].notna(), 'patente_uf']
no_numb.head()


,patente_uf
0,"UF 10,58"
12,"UF 6,47"
20,"UF 13,36"
83,"UF 6,53"
92,UF


In [31]:
# Corregir datos no numéricos
df['patente_uf'] = pd.to_numeric(
                   df['patente_uf'].astype(str)
                                   .str.replace('UF', '', regex = False)
                                   .str.replace(',', '.', regex = False)
                                   .str.strip(), errors = 'coerce'
                   )

In [32]:
# Volver a verificar
no_numb = df.loc[pd.to_numeric(df['patente_uf'], errors = 'coerce').isna()
                 & df['patente_uf'].notna(), 'patente_uf']
no_numb.head()

,patente_uf


### Solución

In [33]:
no_numericos = df.loc[pd.to_numeric(df['patente_uf'], errors='coerce').isna()
                      & df['patente_uf'].notna(), 'patente_uf']
print('Casos no convertibles:', len(no_numericos))
no_numericos.head(8).tolist()

Casos no convertibles: 0


[]

In [34]:
df['patente_uf'] = (df['patente_uf'].astype(str)
                      .str.replace('UF', '', regex=False)
                      .str.replace(',', '.', regex=False)
                      .str.strip())
df['patente_uf'] = pd.to_numeric(df['patente_uf'], errors='coerce')
print('dtype final:', df['patente_uf'].dtype)
df['patente_uf'].describe().round(2)

dtype final: float64


,patente_uf
count,538.00
mean,8.37
std,5.46
min,1.54
25%,4.56
50%,6.85
75%,10.61
max,33.11


---
## Ejercicio 6 · Fechas en formatos mixtos

**Enunciado.** `fecha_patente` viene en cuatro formatos distintos. Conviértela a fecha sin perder registros.

In [37]:
# Tu intento aquí
FORMATOS = ['%Y-%m-%d', '%d-%m-%Y', '%d/%m/%Y', '%Y%m%d']

fechas = pd.Series(pd.NaT, index = df.index)

for fmt in FORMATOS:
  pend = fechas.isna()
  if not pend.any(): break
  fechas.loc[pend] = pd.to_datetime(
                     df.loc[pend, 'fecha_patente'], errors = 'coerce', format = fmt
                     )

df['fecha_patente'] = fechas

In [38]:
print(df['fecha_patente'])

0     2016-06-19
1     2022-10-06
2     2022-05-18
3     2018-08-05
4     2025-07-01
         ...    
585   2021-07-21
586   2017-06-27
587   2024-11-13
588   2019-03-15
589   2025-01-26
Name: fecha_patente, Length: 560, dtype: datetime64[ns]


### Solución

In [39]:
estricto = pd.to_datetime(df['fecha_patente'], errors='coerce', format='%Y-%m-%d')
print('No calzan con %Y-%m-%d:', estricto.isna().sum())
df.loc[estricto.isna(), 'fecha_patente'].head(8).tolist()

No calzan con %Y-%m-%d: 0


[]

In [40]:
# Estrategia: probar formatos en orden, del más específico al más general
FORMATOS = ['%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y', '%Y%m%d']

fechas = pd.Series(pd.NaT, index=df.index)
for fmt in FORMATOS:
    pendientes = fechas.isna()
    if not pendientes.any():
        break
    intento = pd.to_datetime(df.loc[pendientes, 'fecha_patente'],
                             errors='coerce', format=fmt)
    fechas.loc[pendientes] = intento
    print(f'{fmt:12s} -> recupera {intento.notna().sum():3d} · quedan {fechas.isna().sum():3d}')

df['fecha_patente'] = fechas

%Y-%m-%d     -> recupera 560 · quedan   0


**Por qué no usar `dayfirst=True` y listo.** Porque `03-08-2019` es ambiguo: puede ser el 3 de agosto o el 8 de marzo, y pandas elegirá uno sin avisarte. Probar formatos explícitos en orden te obliga a declarar qué supusiste, y eso es lo que va a la bitácora.

> **Decisión razonable:** conversión por formatos explícitos, declarando el supuesto de que el formato `dd-mm-aaaa` es día primero. *Bitácora: Corregir, con supuesto declarado.*

---
## Ejercicio 7 · Unidad inconsistente

**Enunciado.** Hay nueve registros con una superficie de menos de 1 m². Ningún local comercial mide eso. ¿Qué pasó y cómo se arregla?

In [48]:
# Tu intento aquí
print(df['superficie_m2'].head(9))

0    74.8
1    79.1
2    80.9
3    13.2
4    14.3
5    38.7
6    40.6
7    33.9
8    47.8
Name: superficie_m2, dtype: float64


### Solución

In [49]:
sospechosos = df.loc[df['superficie_m2'] < 1,
                     ['establecimiento', 'rubro_norm', 'superficie_m2', 'trabajadores']]
print('Registros con superficie < 1 m2:', len(sospechosos))
sospechosos.head(9)

Registros con superficie < 1 m2: 9


,establecimiento,rubro_norm,superficie_m2,trabajadores
75,La Portada Ltda.,FARMACIA,0.0718,3.0
117,El Molino SpA,RESTAURANTE,0.0296,1.0
169,Los Maitenes Ltda.,PANADERIA,0.1008,5.0
176,El Molino II,TALLER MECANICO,0.0467,1.0
182,Villa Nueva SpA,FERRETERIA,0.0267,2.0
266,Los Aromos Ltda.,PANADERIA,0.0455,1.0
348,El Manzano y Cía.,PELUQUERIA,0.0464,3.0
356,El Roble,TALLER MECANICO,0.0353,1.0
568,Villa Nueva II,ALMACEN,0.0509,1.0


In [50]:
# Si estuvieran en hectáreas, x 10.000 los devuelve a un rango plausible
print('Al convertir a m2:')
print((sospechosos['superficie_m2'] * 10_000).round(1).tolist())

Al convertir a m2:
[718.0, 296.0, 1008.0, 467.0, 267.0, 455.0, 464.0, 353.0, 509.0]


**Lectura.** Multiplicadas por 10.000 caen en el rango normal del resto del conjunto: estaban **cargadas en hectáreas**. Este defecto es especialmente traicionero porque los valores son numéricos, plausibles como número y silenciosos: no rompen nada, solo hunden el promedio.

> **Decisión razonable:** convertir esos nueve registros a m², dejando constancia. Si no tuvieras forma de confirmar la hipótesis, la alternativa honesta sería descartarlos y declararlo. *Bitácora: Corregir, con hipótesis declarada.*

In [51]:
mask = df['superficie_m2'] < 1
df.loc[mask, 'superficie_m2'] = df.loc[mask, 'superficie_m2'] * 10_000
print('Corregidos:', mask.sum())
df['superficie_m2'].describe().round(1)

Corregidos: 9


,superficie_m2
count,550.0
mean,105.3
std,378.4
min,9.1
25%,35.7
50%,50.2
75%,77.4
max,4197.1


---
## Ejercicio 8 · Atípicos: el real y el erróneo

**Enunciado.** Este es el ejercicio más importante del cuaderno. Hay valores extremos en `trabajadores` y en `superficie_m2`. **No todos son errores.** Distingue cuáles corregir y cuáles conservar.

In [52]:
# Tu intento aquí
df.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
fecha_patente,560,2020-06-17 10:45:25.714285568,2015-01-21 00:00:00,2017-08-29 18:00:00,2020-07-21 12:00:00,2023-03-05 06:00:00,2025-09-02 00:00:00,NaN
superficie_m2,550.0,105.255273,9.1,35.725,50.2,77.4,4197.1,378.392694
patente_uf,538.0,8.365613,1.54,4.56,6.85,10.61,33.11,5.461054
trabajadores,545.0,12.4,1.0,1.0,2.0,3.0,1200.0,98.267631


In [53]:
# Número de trabajadores mayor al cuartil 3
q1, q3 = df['trabajadores'].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
df.loc[df['trabajadores'] > limite_superior]

,rol,establecimiento,rubro,comuna,region,fecha_patente,superficie_m2,patente_uf,trabajadores,estado,ultima_fiscalizacion,comuna_norm,rubro_norm
64,02-00529,La Portada II,ferretería,Concepción,Biobío,2023-05-23,103.9,17.48,7.0,Vigente,2025-10-17,CONCEPCION,FERRETERIA
72,01-00028,Costa Azul SpA,Verdulería,Los Ángeles,Biobío,2020-03-04,200.7,33.11,9.0,Vigente,2022-12-13,LOS ANGELES,VERDULERIA
78,01-00454,El Molino SpA,Verdulería,Los Ángeles,Biobío,2016-09-28,121.5,20.17,7.0,Vigente,2021-02-10,LOS ANGELES,VERDULERIA
96,01-00171,Punto Sur Ltda.,Bazar,San Pedro de la Paz,Biobío,2020-01-29,134.9,23.57,8.0,Vigente,2022-02-21,SAN PEDRO DE LA PAZ,BAZAR
102,01-00055,El Rincón y Cía.,Farmacia,San Pedro de la Paz,Biobío,2021-05-17,189.7,26.85,7.0,Vigente,2025-09-18,SAN PEDRO DE LA PAZ,FARMACIA
132,01-00417,Punto Sur,Verdulería,Coronel,Biobío,2017-02-27,121.8,14.49,7.0,Vigente,2022-04-10,CORONEL,VERDULERIA
146,03-00015,La Bahía y Cía.,Ferretería,Chillán,Ñuble,2019-11-23,198.1,30.08,9.0,Vigente,2025-07-18,CHILLAN,FERRETERIA
168,03-00006,El Rincón II,Almacén,Concepcion,Biobío,2016-05-04,2712.6,3.86,70.0,Vigente,2019-04-15,CONCEPCION,ALMACEN
217,03-00325,Ñielol SpA,Bazar,Concepción,Biobío,2021-11-02,48.3,8.30,1200.0,Vigente,2026-08-04,CONCEPCION,BAZAR
236,03-00255,El Roble Ltda.,Taller mecánico,Coronel,Biobío,2024-04-02,15.7,1.91,1200.0,Caducada,2026-01-15,CORONEL,TALLER MECANICO


### Solución

In [54]:
# Criterio clásico: rango intercuartílico
q1, q3 = df['trabajadores'].quantile([0.25, 0.75])
iqr = q3 - q1
limite = q3 + 1.5 * iqr
print(f'Q1={q1:.0f}  Q3={q3:.0f}  IQR={iqr:.0f}  límite superior={limite:.0f}')
df.loc[df['trabajadores'] > limite,
       ['establecimiento', 'rubro_norm', 'superficie_m2', 'trabajadores']].sort_values('trabajadores')

Q1=1  Q3=3  IQR=2  límite superior=6


,establecimiento,rubro_norm,superficie_m2,trabajadores
64,La Portada II,FERRETERIA,103.9,7.0
78,El Molino SpA,VERDULERIA,121.5,7.0
132,Punto Sur,VERDULERIA,121.8,7.0
102,El Rincón y Cía.,FARMACIA,189.7,7.0
481,Villa Nueva Ltda.,TALLER MECANICO,237.1,7.0
512,Ñielol Ltda.,PELUQUERIA,186.1,7.0
352,El Rincón II,PANADERIA,184.9,7.0
335,Las Camelias,PELUQUERIA,100.6,7.0
566,El Rincón SpA,FERRETERIA,121.9,7.0
96,Punto Sur Ltda.,BAZAR,134.9,8.0


Ahora la pregunta que ninguna fórmula responde: **¿cuáles son reales?**

El criterio no es estadístico, es de dominio. Cruza `trabajadores` con `superficie_m2`: un local de 134 m² con 400 trabajadores es imposible; uno de 3.000 m² con 60 es un supermercado.

In [55]:
df['m2_por_trabajador'] = df['superficie_m2'] / df['trabajadores']

print('IMPLAUSIBLES (menos de 2 m2 por trabajador):')
display(df.loc[df['m2_por_trabajador'] < 2,
               ['establecimiento', 'superficie_m2', 'trabajadores', 'm2_por_trabajador']]
          .round(2))

print('\nPLAUSIBLES aunque extremos (locales grandes):')
display(df.loc[(df['superficie_m2'] > 2000) & (df['m2_por_trabajador'] >= 2),
               ['establecimiento', 'rubro_norm', 'superficie_m2', 'trabajadores']]
          .round(1))

IMPLAUSIBLES (menos de 2 m2 por trabajador):


,establecimiento,superficie_m2,trabajadores,m2_por_trabajador
217,Ñielol SpA,48.3,1200.0,0.04
236,El Roble Ltda.,15.7,1200.0,0.01
273,Ñielol Ltda.,59.6,1200.0,0.05
279,Ñielol y Cía.,49.9,700.0,0.07
417,El Roble II,176.3,700.0,0.25



PLAUSIBLES aunque extremos (locales grandes):


,establecimiento,rubro_norm,superficie_m2,trabajadores
168,El Rincón II,ALMACEN,2712.6,70.0
253,La Esquina Ltda.,TALLER MECANICO,4197.1,84.0
283,Ñielol,ALMACEN,3965.7,72.0
310,El Manzano SpA,FARMACIA,4122.6,76.0
505,Villa Nueva SpA,FERRETERIA,3583.8,75.0
543,Los Maitenes II,FARMACIA,2972.9,58.0


**Lectura.** El rango intercuartílico marca los dos grupos como atípicos. Pero:

- Los de **menos de 2 m² por trabajador** son un error de digitación (sobra un cero). Se corrigen.
- Los **locales grandes con proporción razonable** son supermercados reales. Se conservan.

> **Si hubieras eliminado todos los atípicos automáticamente**, habrías borrado exactamente los establecimientos más interesantes del conjunto. Eliminar atípicos «porque son atípicos» es una de las formas más eficientes de destruir información.

*Bitácora: dos entradas distintas —Corregir para los de digitación, ninguna acción para los reales.*

In [56]:
mask = df['m2_por_trabajador'] < 2
df.loc[mask, 'trabajadores'] = (df.loc[mask, 'trabajadores'] / 10).round()
print('Corregidos por digitación:', mask.sum())
df['trabajadores'].describe().round(1)

Corregidos por digitación: 5


,trabajadores
count,545.0
mean,4.1
std,12.2
min,1.0
25%,1.0
50%,2.0
75%,3.0
max,120.0


---
## Cierre · El flujo completo como función reutilizable

Todo lo anterior, junto y en orden. Así es como se ve una rutina de limpieza profesional: **una función, con la semilla fijada y cada paso justificado en la bitácora.**

In [57]:
def limpiar_patentes(ruta, semilla=SEMILLA):
    """Flujo de limpieza del registro de patentes. Devuelve (df_limpio, informe)."""
    np.random.seed(semilla)
    d = pd.read_csv(ruta, sep=';')
    informe = {'filas_iniciales': len(d)}

    # 1 · centinelas -> faltantes
    d['trabajadores']  = pd.to_numeric(d['trabajadores'], errors='coerce').replace(-1, np.nan)
    d['superficie_m2'] = pd.to_numeric(d['superficie_m2'], errors='coerce').replace(9999, np.nan)
    d['ultima_fiscalizacion'] = d['ultima_fiscalizacion'].replace('S/I', np.nan)

    # 2 · categorías
    d['comuna_norm'] = normalizar_texto(d['comuna'])
    d['rubro_norm']  = normalizar_texto(d['rubro']).replace(EQUIVALENCIAS)

    # 3 · duplicados sobre clave normalizada
    k = (d['rol'].astype(str).str.replace(' ', '', regex=False) + '|' +
         normalizar_texto(d['establecimiento']))
    d = d.loc[~k.duplicated()].copy()
    informe['duplicados_eliminados'] = informe['filas_iniciales'] - len(d)

    # 4 · numérico como texto
    d['patente_uf'] = pd.to_numeric(
        d['patente_uf'].astype(str).str.replace('UF', '', regex=False)
                       .str.replace(',', '.', regex=False).str.strip(),
        errors='coerce')

    # 5 · fechas por formatos explícitos
    f = pd.Series(pd.NaT, index=d.index)
    for fmt in FORMATOS:
        p = f.isna()
        if not p.any():
            break
        f.loc[p] = pd.to_datetime(d.loc[p, 'fecha_patente'], errors='coerce', format=fmt)
    d['fecha_patente'] = f

    # 6 · unidad inconsistente (hectáreas -> m2)
    m = d['superficie_m2'] < 1
    d.loc[m, 'superficie_m2'] *= 10_000
    informe['superficies_reescaladas'] = int(m.sum())

    # 7 · atípicos por digitación (no los reales)
    d['m2_por_trabajador'] = d['superficie_m2'] / d['trabajadores']
    m = d['m2_por_trabajador'] < 2
    d.loc[m, 'trabajadores'] = (d.loc[m, 'trabajadores'] / 10).round()
    informe['atipicos_corregidos'] = int(m.sum())

    informe['filas_finales'] = len(d)
    return d, informe


limpio, informe = limpiar_patentes('S2_Dataset_Ejercicios_CDD.csv')
for k, v in informe.items():
    print(f'{k:26s}: {v}')

filas_iniciales           : 590
duplicados_eliminados     : 30
superficies_reescaladas   : 9
atipicos_corregidos       : 5
filas_finales             : 560


### Lo que deberías llevarte

1. **Los tipos de dato son el primer diagnóstico**: una columna numérica como texto siempre esconde algo.
2. **Los centinelas son invisibles en los tipos**: solo aparecen mirando rangos.
3. **Un faltante que se concentra en un grupo es informativo**, y no se imputa.
4. **Los casi-duplicados sobreviven a `drop_duplicates()`**: hay que normalizar la clave primero.
5. **No todo atípico es un error.** Distinguirlos exige conocer el dominio, no una fórmula.
6. **Cada decisión va a la bitácora**, con su justificación y su efecto esperado.

Ahora aplica este mismo recorrido sobre **tu propio conjunto de datos**, en el Laboratorio 2.

_Ciencia de Datos · Advance UCSC · Semana 2_